# Artemis Load Balancer - Analysis Template

This notebook provides a template for analyzing load balancer experiment results.

## What this notebook does:
1. Load experiment results from CSV logs
2. Compute SLA metrics using `sla_monitor`
3. Visualize latency, cost, and model usage
4. Compare across load profiles
5. Deep-dive into specific task types or models

## Related notebooks:
- For raw dataset analysis, see `artemis_final/ares/notebooks/`
- For router evaluation, see router notebooks
- For per-task/model stats validation, see Ares aggregation notebooks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add load_balancer to path
sys.path.insert(0, str(Path.cwd().parent.parent))

from load_balancer.sla_monitor import compute_sla_metrics, compute_detailed_metrics, print_detailed_summary
from load_balancer.metrics_logger import load_decisions_from_csv

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Experiment Data

In [ ]:
# Configure experiment to analyze
EXPERIMENT_NAME = "phase5_dynamic_load"  # Change this to your experiment name
OUTPUT_DIR = Path(f"../outputs/{EXPERIMENT_NAME}")
CSV_PATH = OUTPUT_DIR / "decisions.csv"

# Load CSV data
df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} decisions from {CSV_PATH}")
print(f"\nExperiment: {df['experiment_name'].iloc[0]}")
print(f"Load profiles: {df['load_profile'].unique().tolist()}")
print(f"Models: {df['chosen_model'].unique().tolist()}")
print(f"Tasks: {df['task_type'].unique().tolist()}")

df.head()

## 2. Overall Metrics

In [ ]:
# Load as SchedulingDecision objects for metric computation
decisions = load_decisions_from_csv(CSV_PATH)

# Compute detailed metrics
LATENCY_SLA_MS = 2000.0  # Adjust to your experiment's SLA

load_profile_map = dict(zip(df['sample_id'], df['load_profile']))
detailed_metrics = compute_detailed_metrics(
    decisions,
    LATENCY_SLA_MS,
    load_profile_map
)

# Print summary
print_detailed_summary(detailed_metrics)

## 3. Latency Analysis

In [ ]:
# Latency distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall latency distribution
axes[0].hist(df['total_latency_ms'], bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(LATENCY_SLA_MS, color='red', linestyle='--', linewidth=2, label='SLA')
axes[0].set_xlabel('Total Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Overall Latency Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Latency by load profile
for profile in df['load_profile'].unique():
    profile_df = df[df['load_profile'] == profile]
    axes[1].hist(profile_df['total_latency_ms'], bins=30, alpha=0.5, label=profile)

axes[1].axvline(LATENCY_SLA_MS, color='red', linestyle='--', linewidth=2, label='SLA')
axes[1].set_xlabel('Total Latency (ms)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Latency Distribution by Load Profile')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Latency breakdown: queue vs service time
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: queue delay vs service time
axes[0].scatter(df['queue_delay_ms'], df['service_time_ms'], alpha=0.3, s=20)
axes[0].set_xlabel('Queue Delay (ms)')
axes[0].set_ylabel('Service Time (ms)')
axes[0].set_title('Queue Delay vs Service Time')
axes[0].grid(True, alpha=0.3)

# Box plot by load profile
profile_data = []
profile_labels = []
for profile in df['load_profile'].unique():
    profile_df = df[df['load_profile'] == profile]
    profile_data.append(profile_df['total_latency_ms'])
    profile_labels.append(profile)

axes[1].boxplot(profile_data, labels=profile_labels)
axes[1].axhline(LATENCY_SLA_MS, color='red', linestyle='--', linewidth=2, label='SLA')
axes[1].set_ylabel('Total Latency (ms)')
axes[1].set_title('Latency by Load Profile (Box Plot)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Cost Analysis

In [ ]:
# Cost distribution and totals
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cost histogram
axes[0].hist(df['est_cost_usd'], bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Cost per Request (USD)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Cost Distribution')
axes[0].grid(True, alpha=0.3)

# Total cost by load profile
cost_by_profile = df.groupby('load_profile')['est_cost_usd'].sum()
axes[1].bar(cost_by_profile.index, cost_by_profile.values)
axes[1].set_xlabel('Load Profile')
axes[1].set_ylabel('Total Cost (USD)')
axes[1].set_title('Total Cost by Load Profile')
axes[1].grid(True, alpha=0.3)

# Add values on bars
for i, v in enumerate(cost_by_profile.values):
    axes[1].text(i, v, f'${v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nTotal cost across all requests: ${df['est_cost_usd'].sum():.4f}")
print(f"Average cost per request: ${df['est_cost_usd'].mean():.6f}")

## 5. Model Usage Analysis

In [ ]:
# Model usage distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall model usage
model_counts = df['chosen_model'].value_counts()
axes[0].pie(model_counts.values, labels=model_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Overall Model Usage')

# Model usage by load profile
usage_by_profile = pd.crosstab(df['load_profile'], df['chosen_model'], normalize='index') * 100
usage_by_profile.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_xlabel('Load Profile')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Model Usage by Load Profile')
axes[1].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Router vs chosen model comparison
model_switch_rate = (df['chosen_model'] != df['preferred_model']).mean()
print(f"Model switch rate: {model_switch_rate:.2%}")
print(f"  (Fraction of requests where chosen model differs from router's preference)")

# Breakdown by load profile
switch_by_profile = df.groupby('load_profile').apply(
    lambda x: (x['chosen_model'] != x['preferred_model']).mean()
)

fig, ax = plt.subplots(figsize=(10, 5))
switch_by_profile.plot(kind='bar', ax=ax)
ax.set_ylabel('Model Switch Rate')
ax.set_title('Model Switch Rate by Load Profile')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

# Add values on bars
for i, v in enumerate(switch_by_profile.values):
    ax.text(i, v, f'{v:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. SLA Violation Analysis

In [ ]:
# SLA violation rates
violation_rate = df['sla_violated'].mean()
print(f"Overall SLA violation rate: {violation_rate:.2%}")

# By load profile
violation_by_profile = df.groupby('load_profile')['sla_violated'].mean()
print("\nSLA violation rate by load profile:")
for profile, rate in violation_by_profile.items():
    print(f"  {profile:15s}: {rate:.2%}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
violation_by_profile.plot(kind='bar', ax=ax, color='coral')
ax.set_ylabel('SLA Violation Rate')
ax.set_title('SLA Violation Rate by Load Profile')
ax.set_ylim([0, max(1, violation_by_profile.max() * 1.2)])
ax.axhline(0.05, color='red', linestyle='--', linewidth=2, label='5% Target')
ax.legend()
ax.grid(True, alpha=0.3)

# Add values on bars
for i, v in enumerate(violation_by_profile.values):
    ax.text(i, v, f'{v:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 7. Accuracy Analysis

In [ ]:
# Accuracy distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall accuracy
axes[0].hist(df['est_accuracy'], bins=30, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Estimated Accuracy')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Accuracy Distribution')
axes[0].grid(True, alpha=0.3)

# Accuracy drop distribution
axes[1].hist(df['accuracy_drop'], bins=30, alpha=0.7, edgecolor='black', color='orange')
axes[1].set_xlabel('Accuracy Drop vs Preferred Model')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Accuracy Drop Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage accuracy: {df['est_accuracy'].mean():.4f}")
print(f"Average accuracy drop: {df['accuracy_drop'].mean():.4f}")
print(f"Max accuracy drop: {df['accuracy_drop'].max():.4f}")

## 8. Per-Task Analysis

In [ ]:
# Metrics by task type
task_metrics = df.groupby('task_type').agg({
    'total_latency_ms': ['mean', 'median', lambda x: x.quantile(0.95)],
    'est_cost_usd': 'mean',
    'est_accuracy': 'mean',
    'sla_violated': 'mean',
    'sample_id': 'count'
}).round(4)

task_metrics.columns = ['Avg Latency', 'Median Latency', 'P95 Latency', 
                         'Avg Cost', 'Avg Accuracy', 'Violation Rate', 'Count']
print("\nMetrics by Task Type:")
print(task_metrics)

## 9. Cost vs Latency Trade-off

In [ ]:
# Scatter plot: cost vs latency, colored by model
fig, ax = plt.subplots(figsize=(12, 6))

for model in df['chosen_model'].unique():
    model_df = df[df['chosen_model'] == model]
    ax.scatter(
        model_df['total_latency_ms'],
        model_df['est_cost_usd'],
        label=model,
        alpha=0.5,
        s=30
    )

ax.axvline(LATENCY_SLA_MS, color='red', linestyle='--', linewidth=2, alpha=0.7, label='SLA')
ax.set_xlabel('Total Latency (ms)')
ax.set_ylabel('Cost (USD)')
ax.set_title('Cost vs Latency Trade-off')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Autoscaling Analysis

In [ ]:
# Number of replicas over time
fig, ax = plt.subplots(figsize=(15, 5))

for model in df['chosen_model'].unique():
    model_df = df[df['chosen_model'] == model].sort_values('global_step')
    ax.plot(model_df['global_step'], model_df['num_replicas'], label=model, alpha=0.7)

ax.set_xlabel('Global Step')
ax.set_ylabel('Number of Replicas')
ax.set_title('Autoscaling: Replicas Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Comparison with Baseline (Optional)

If you have results from multiple experiments (e.g., different scheduling modes),
you can load and compare them here.

In [ ]:
# Example: Compare capacity_aware vs router_only
# BASELINE_CSV = OUTPUT_DIR / "../router_only_baseline/decisions.csv"
# baseline_df = pd.read_csv(BASELINE_CSV)

# # Compare violation rates
# print("Violation Rate Comparison:")
# print(f"  Capacity-aware: {df['sla_violated'].mean():.2%}")
# print(f"  Router-only:    {baseline_df['sla_violated'].mean():.2%}")

## 12. Export Summary Statistics

In [ ]:
# Create summary report
summary = {
    "experiment_name": df['experiment_name'].iloc[0],
    "total_requests": len(df),
    "load_profiles": df['load_profile'].unique().tolist(),
    "models_used": df['chosen_model'].unique().tolist(),
    "overall_metrics": {
        "latency_p50_ms": df['total_latency_ms'].quantile(0.5),
        "latency_p95_ms": df['total_latency_ms'].quantile(0.95),
        "latency_p99_ms": df['total_latency_ms'].quantile(0.99),
        "violation_rate": df['sla_violated'].mean(),
        "avg_cost_usd": df['est_cost_usd'].mean(),
        "total_cost_usd": df['est_cost_usd'].sum(),
        "avg_accuracy": df['est_accuracy'].mean(),
        "model_switch_rate": (df['chosen_model'] != df['preferred_model']).mean(),
    }
}

print("\n" + "="*60)
print("SUMMARY REPORT")
print("="*60)
for key, value in summary.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

# Optionally save to JSON
import json
summary_path = OUTPUT_DIR / "analysis_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved to: {summary_path}")

## Next Steps

### For deeper analysis:
1. **Validate per-task/model stats**: Open Ares notebooks to see how latency, cost, and accuracy were computed
2. **Compare routing strategies**: Run experiments with different scheduling modes and compare results
3. **Tune autoscaling**: Adjust `capacity_config.yaml` and re-run experiments
4. **Analyze specific failure cases**: Filter for high-latency or low-accuracy requests and investigate

### Related notebooks:
- `artemis_final/ares/notebooks/` - Dataset analysis and stats computation
- Router evaluation notebooks - Compare router predictions with actual performance